# Hybrid Quantum-Classical Reinforcement Learning (2/3)

___
___

## Introduction
___

This series of experiments is based on the 2025 paper by Nagy et al.: "[Hybrid Quantum-Classical Reinforcement Learning in Latent Observation Spaces](https://arxiv.org/abs/2410.18284)".

In this article, the authors apply hybrid quantum-classical Reinforcement Learning (RL) models to two simulated environments. They compare classical, qubit-based and photonic-based agents, all using Proximal Policy Optimization (PPO). They also use an AutoEncoder (AE) to compress the dimensionality of the observations and train that AE jointly with the PPO agents.

The notebook series aims to compare the resources cost of the different systems to reach the same performance. It is divided in three parts:

- [Part I: Classical vs Qubit agents on the Cart Pole environment](QRL_experiment_1.ipynb)
- **Part II: Classical vs Qubit agents on the Lunar Lander and Maze environments**
- [Part III: Photonic agents on the three environments](QRL_experiment_3.ipynb)

This second notebook will present:
1. The Lunar Lander environment
2. The custom Maze environment
3. Convolutional Neural Networks (CNNs)
4. Results
5. Conclusions
6. Follow-up

In [1]:
# Uncomment the following line to install dependencies if needed
# !pip install ipynb torch torchvision gymnasium swig "gymnasium[box2d]" matplotlib pennylane

In [2]:
import os
import random
from itertools import count
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym
from MazeEnvironment import Maze

from ipynb.fs.defs.QRL_experiment_1 import AutoEncoder, EncoderDecoderNN, PPO, ActorNN, CriticNN, ActorQubitNN, MemoryTracker, count_parameters, save_results, plot_results

# For reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "cpu"
)

## 1 - The Lunar Lander environment
___

In this Box2D environment, the agent learns to land a spaceship in a designated area. 

<div align="center">
<img src="images/qrl_demo_lunarlander.gif" width="300"/>
</div>

At each step, the agent gets 8 observations: the x and y coordinates of the ship, its x and y linear velocities, its angle, angular velocity, and whether each leg of the ship is touching the ground. The agent can choose between 4 actions: do nothing, fire the main engine, fire the left engine or fire the right one.

A reward is granted at each step, depending on the ship's position compared to the landing pad, its velocity, orientation, engine firings and contact with the ground. An additional reward is given at the end depending on whether the ship landed correctly or not. The episode ends when the ship crashes, is getting out of display or stalls.

More information about this environment can be found here: https://gymnasium.farama.org/environments/box2d/lunar_lander/.

In [3]:
# Create the Lunar Lander environment with gymnasium
lunarlander_env = gym.make("LunarLander-v3", render_mode="rgb_array")
lunarlander_env.reset(seed=seed)
lunarlander_env.action_space.seed(seed)
lunarlander_env.observation_space.seed(seed)

42

## 2 - The Maze environment
___

In this custom environment, the agent (the blue square) learns to navigate a maze to find the exit (the green square). 

<div align="center">
<img src="images/qrl_demo_maze.gif" width="300"/>
</div>

At each step, the agent gets a 96x96 RGB image as observations. It can then choose between 4 actions: go up, go down, go right or go left.

The reward is +1 when the agent gets to the exit, -1 if it hits a wall. The episode ends if the agent hits a wall, gets to the exit or if it spent too much time exploring (if it has moved more times than there are squares in the grid). 

The code for this environment can be found here: [MazeEnvironment.py](MazeEnvironment.py).

In [4]:
# Create the custom Maze environment
maze_env = Maze()
maze_env.reset(seed=seed)
maze_env.action_space.seed(seed)
maze_env.observation_space.seed(seed)

42

## 3 - Convolutional Neural Networks (CNNs)
___

The first notebook introduced AutoEncoders, as well as classical and qubit-based Proximal Policy Optimization (PPO) agents.

For the Maze environment however, since the input is an image, we will use a Convolutional Neural Network (CNN) as the first part of the AutoEncoder.

Convolutional layers apply small filters across the image to detect local patterns such as edges, corners, and shapes while preserving the spatial relationships between pixels. By applying multiple convolutional layers, the model learns to encode increasingly abstract visual features.

<div align="center">
<img src="images/qrl_demo_cnn.png" width="600"/>
</div>

In this notebook, the CNN is created from scratch and pretrained on images from the Maze environment. Then, the convolutional layers are frozen for the actual PPO training, which helps training stability. You can also use a pretrained model instead, like ResNet.

In [5]:
class EncoderCNN(nn.Module):
    def __init__(self, input_shape, hidden_dims, kernel_size=4, stride_size=2, padding=1):
        super().__init__()
        width, _, input_channels = input_shape
        self.conv1 = nn.Conv2d(input_channels, hidden_dims[0], kernel_size, stride_size, padding)
        conv1_size = int((width+2*padding-kernel_size)/stride_size+1)
        self.conv2 = nn.Conv2d(hidden_dims[0], hidden_dims[1], kernel_size, stride_size, padding)
        conv2_size = int((conv1_size+2*padding-kernel_size)/stride_size+1)
        self.conv3 = nn.Conv2d(hidden_dims[1], hidden_dims[2], kernel_size, stride_size, padding)
        conv3_size = int((conv2_size+2*padding-kernel_size)/stride_size+1)
        self.flatten = nn.Flatten()
        self.output_size = hidden_dims[2] * conv3_size**2

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.flatten(x)
        return x

class DecoderCNN(nn.Module):
    def __init__(self, output_shape, hidden_dims, kernel_size=4, stride_size=2, padding=1):
        super().__init__()
        width, _, output_channels = output_shape
        self.conv1 = nn.ConvTranspose2d(hidden_dims[0], output_channels, kernel_size, stride_size, padding)
        conv1_size = int((width+2*padding-kernel_size)/stride_size+1)
        self.conv2 = nn.ConvTranspose2d(hidden_dims[1], hidden_dims[0], kernel_size, stride_size, padding)
        conv2_size = int((conv1_size+2*padding-kernel_size)/stride_size+1)
        self.conv3 = nn.ConvTranspose2d(hidden_dims[2], hidden_dims[1], kernel_size, stride_size, padding)
        conv3_size = int((conv2_size+2*padding-kernel_size)/stride_size+1)
        self.intermediate_shape = (-1, hidden_dims[2], conv3_size, conv3_size)
        
    def forward(self, x):
        x = x.view(self.intermediate_shape)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv2(x))
        x = F.sigmoid(self.conv1(x))
        return x.permute(0, 2, 3, 1)

class CriticCNN(nn.Module):
    def __init__(self, input_shape, intermediate_dim, output_dim):
        super().__init__()
        self.cnn = EncoderCNN(input_shape, intermediate_dim)
        self.nn = CriticNN(self.cnn.output_size, intermediate_dim[-1], output_dim)

    def forward(self, x):
        x = self.cnn(x)
        x = self.nn(x)
        return x

class ConvolutionalAE(nn.Module):
    def __init__(self, input_shape, parameters):
        super().__init__()
        self.image_shape = input_shape
        self.cnn_encoder = EncoderCNN(self.image_shape, parameters["ae_hidden_dims"])
        self.cnn_decoder = DecoderCNN(self.image_shape, parameters["ae_hidden_dims"])
        self.encoder = nn.Sequential(
            self.cnn_encoder,
            EncoderDecoderNN(self.cnn_encoder.output_size, parameters["ae_hidden_dims"][-1], parameters["ae_output_dim"])
        )
        self.decoder = nn.Sequential(
            EncoderDecoderNN(parameters["ae_output_dim"], parameters["ae_hidden_dims"][-1], self.cnn_encoder.output_size),
            self.cnn_decoder
        )
        self.hyperparameters = parameters

    def preprocess(self, state):
        state = state / 255.0
        return state
    
    def pre_train(self, env, device):
        print("Pre-training ConvolutionalAE")
        optimizer = optim.Adam(self.parameters())
        epochs = self.hyperparameters["ae_pretrain_epochs"]
        batch_size = self.hyperparameters["ae_pretrain_batchsize"]

        states = []
        while len(states) < self.hyperparameters["ae_pretrain_n_examples"]:
            state, _ = env.reset()
            for t in count():
                state = self.preprocess(state)
                state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
                states.append(state)

                action = env.action_space.sample()
                state, _, terminated, truncated, _ = env.step(action)

                if terminated or truncated:
                    break
        states = torch.cat(states)

        print(f"Collected {states.shape[0]} images for training")
        
        for epoch in range(epochs):
            perm = torch.randperm(len(states))
            shuffled = states[perm]
            for i in range(0, len(shuffled) // batch_size + 1, batch_size):
                x = shuffled[i:i+batch_size]

                x_hat = self.decoder(F.tanh(self.encoder(x)))
                loss = F.mse_loss(x_hat, x).mean()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            if (epoch+1)%(epochs//10) == 0:
                title = f"Epoch {epoch+1}: loss = {loss:.4f}"
                x = x.detach().cpu().numpy()
                x = (x*255.0).astype(np.uint8)

                x_hat = x_hat.detach().cpu().numpy()
                x_hat = (x_hat*255.0).astype(np.uint8)

                _, axes = plt.subplots(1, 2)
                axes[0].imshow(x[0])
                axes[1].imshow(x_hat[0])
                plt.suptitle(title)
                plt.tight_layout()
                plt.show()
            

In [6]:
def freeze_conv_layers(module):
    for m in module.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            for p in m.parameters():
                p.requires_grad = False

class MazePPO(PPO):
    def __init__(self, *args, results_folder, **kwargs):
        super().__init__(*args, **kwargs)
        autoencoder_path = os.path.join(results_folder, "cnn_ae.pt")
        if not os.path.exists(autoencoder_path):
            self.autoencoder.pre_train(self.env, self.device)
            torch.save(self.autoencoder.state_dict(), autoencoder_path)
        else:
            print("Loading ae weights")
            self.autoencoder.load_state_dict(torch.load(autoencoder_path, weights_only=True))

        freeze_conv_layers(self.autoencoder)

        self.critic.cnn.load_state_dict(self.autoencoder.cnn_encoder.state_dict())
        freeze_conv_layers(self.critic.cnn)

        self.optimizer = optim.Adam(
            list(filter(lambda p: p.requires_grad, self.autoencoder.parameters())) +
            list(self.actor.parameters()) +
            list(filter(lambda p: p.requires_grad, self.critic.parameters())),
            lr=self.config["lr"]
        )


## 4 - Results
___

In [7]:
results_folder = "results"
os.makedirs(results_folder, exist_ok=True)
results_filepath = os.path.join(results_folder, "qrl_results.json")

### Lunar Lander

In [8]:
env_name = "Lunar Lander"
ll_config = {
    "ae_hidden_dim": 64, "ae_output_dim": 3,
    "critic_intermediate_dim": 128,
    "minibatch_size": 64, "lr": 3e-4,
    "K": 4, "episode_update_frequency": 1, "mean_reward_lookback": 50, "mean_reward_stop": 220,
    "gamma": 0.99, "lambda": 0.98, "epsilon": 0.2, "entropy_coeff": 0.01,
    }

In [ ]:
# 1. Classical PPO
ll_config["actor_intermediate_dim"] = 3
with MemoryTracker() as ll_classical_stats:
    ll_classical_ppo = PPO(AutoEncoderClass=AutoEncoder, ActorClass=ActorNN, CriticClass=CriticNN, env=lunarlander_env, config=ll_config, device=device)
    ll_classical_parameters = count_parameters(ll_classical_ppo.actor)
    ll_classical_mean_rewards, ll_classical_training_time = ll_classical_ppo.run()
classical_results = { 
    "training_time": ll_classical_training_time, 
    "mean_rewards": ll_classical_mean_rewards, 
    "parameters": ll_classical_parameters, 
    "memory": ll_classical_stats.peak_rss_mb 
}
torch.save(ll_classical_ppo.autoencoder.state_dict(), os.path.join(results_folder, "ll_classical_ae.pt"))
torch.save(ll_classical_ppo.actor.state_dict(), os.path.join(results_folder, "ll_classical_actor.pt"))
torch.save(ll_classical_ppo.critic.state_dict(), os.path.join(results_folder, "ll_classical_critic.pt"))
qrl_results = save_results(results_filepath, env_name, "classical", classical_results)


Step 1000: mean reward = -216.85 (0:00:00.414788)

Step 2000: mean reward = -214.24 (0:00:00.352442)

Step 3000: mean reward = -223.91 (0:00:00.360109)

Step 4000: mean reward = -225.80 (0:00:00.368647)

Step 5000: mean reward = -234.02 (0:00:00.381702)

Step 6000: mean reward = -221.65 (0:00:00.363491)

Step 7000: mean reward = -208.75 (0:00:00.351361)

Step 8000: mean reward = -202.28 (0:00:00.365701)

Step 9000: mean reward = -174.75 (0:00:00.381811)

Step 10000: mean reward = -171.68 (0:00:00.363631)

Step 11000: mean reward = -172.75 (0:00:00.355561)

Step 12000: mean reward = -184.23 (0:00:00.358442)

Step 13000: mean reward = -184.40 (0:00:00.350246)

Step 14000: mean reward = -183.05 (0:00:00.352371)

Step 15000: mean reward = -185.54 (0:00:00.360138)

Step 16000: mean reward = -191.91 (0:00:00.355344)

Step 17000: mean reward = -191.13 (0:00:00.360319)

Step 18000: mean reward = -191.21 (0:00:00.353642)

Step 19000: mean reward = -181.52 (0:00:00.354417)

Step 20000: mean rew

In [ ]:
ll_classical_ppo.evaluate()

In [ ]:
# 2. Qubit PPO
ll_config["actor_intermediate_dim"] = 4
with MemoryTracker() as ll_qubit_stats:
    ll_qubit_ppo = PPO(AutoEncoderClass=AutoEncoder, ActorClass=ActorQubitNN, CriticClass=CriticNN, env=lunarlander_env, config=ll_config, device=device)
    ll_qubit_parameters = count_parameters(ll_qubit_ppo.actor)
    ll_qubit_mean_rewards, ll_qubit_training_time = ll_qubit_ppo.run()
qubit_results = { 
    "training_time": ll_qubit_training_time, 
    "mean_rewards": ll_qubit_mean_rewards, 
    "parameters": ll_qubit_parameters, 
    "memory": ll_qubit_stats.peak_rss_mb
}
torch.save(ll_qubit_ppo.autoencoder.state_dict(), os.path.join(results_folder, "ll_qubit_ae.pt"))
torch.save(ll_qubit_ppo.actor.state_dict(), os.path.join(results_folder, "ll_qubit_actor.pt"))
torch.save(ll_qubit_ppo.critic.state_dict(), os.path.join(results_folder, "ll_qubit_critic.pt"))
qrl_results = save_results(results_filepath, env_name, "qubit", qubit_results)

In [ ]:
ll_qubit_ppo.evaluate()

In [ ]:
plot_results(env_name, qrl_results[env_name], ll_config["mean_reward_stop"])

### Maze

In [ ]:
env_name = "Maze"
m_config = {
    "image_size": 64,
    "ae_hidden_dims": [16, 32, 64, 128], "ae_output_dim": 8, 
    "ae_pretrain_epochs": 1000, "ae_pretrain_n_examples": 100000, "ae_pretrain_batchsize": 64,
    "critic_intermediate_dim": [16, 32, 64, 128],
    "minibatch_size": 128, "lr": 1e-4,
    "K": 8, "episode_update_frequency": 4, "mean_reward_lookback": 30, "mean_reward_stop": 0.9,
    "gamma": 0.99, "lambda": 0.95, "epsilon": 0.2, "entropy_coeff": 0.01
    }
custom_obs_shape = (m_config["image_size"], m_config["image_size"], 3)

In [ ]:
# 1. Classical PPO
m_config["actor_intermediate_dim"] = 6
with MemoryTracker() as m_classical_stats:
    m_classical_ppo = MazePPO(AutoEncoderClass=ConvolutionalAE, ActorClass=ActorNN, CriticClass=CriticCNN, env=maze_env, config=m_config, device=device, custom_obs_shape=custom_obs_shape, results_folder=results_folder)
    m_classical_parameters = count_parameters(m_classical_ppo.actor)
    m_classical_mean_rewards, m_classical_training_time = m_classical_ppo.run()
classical_results = { 
    "training_time": m_classical_training_time, 
    "mean_rewards": m_classical_mean_rewards, 
    "parameters": m_classical_parameters, 
    "memory": m_classical_stats.peak_rss_mb 
}
torch.save(m_classical_ppo.autoencoder.state_dict(), os.path.join(results_folder, "m_classical_ae.pt"))
torch.save(m_classical_ppo.actor.state_dict(), os.path.join(results_folder, "m_classical_actor.pt"))
torch.save(m_classical_ppo.critic.state_dict(), os.path.join(results_folder, "m_classical_critic.pt"))
qrl_results = save_results(results_filepath, env_name, "classical", classical_results)

In [ ]:
m_classical_ppo.evaluate()

In [ ]:
# 2. Qubit PPO
m_config["actor_intermediate_dim"] = 5
with MemoryTracker() as m_qubit_stats:
    m_qubit_ppo = MazePPO(AutoEncoderClass=ConvolutionalAE, ActorClass=ActorQubitNN, CriticClass=CriticCNN, env=maze_env, config=m_config, device=device, results_folder=results_folder, custom_obs_shape=custom_obs_shape)
    m_qubit_parameters = count_parameters(m_qubit_ppo.actor)
    m_qubit_mean_rewards, m_qubit_training_time = m_qubit_ppo.run()
qubit_results = { 
    "training_time": m_qubit_training_time, 
    "mean_rewards": m_qubit_mean_rewards, 
    "parameters": m_qubit_parameters, 
    "memory": m_qubit_stats.peak_rss_mb
}
torch.save(m_qubit_ppo.autoencoder.state_dict(), os.path.join(results_folder, "m_qubit_ae.pt"))
torch.save(m_qubit_ppo.actor.state_dict(), os.path.join(results_folder, "m_qubit_actor.pt"))
torch.save(m_qubit_ppo.critic.state_dict(), os.path.join(results_folder, "m_qubit_critic.pt"))
qrl_results = save_results(results_filepath, env_name, "qubit", qubit_results)

In [ ]:
m_qubit_ppo.evaluate()

In [ ]:
plot_results(env_name, qrl_results[env_name], m_config["mean_reward_stop"])

## 5 - Conclusions
___

Bigger environments -> Time constraint but how do performances, memory and params follow? For both classical and qubits?

## 6 - Follow-up
___

Again, Try using other values for actor_intermediate_dim, max stop and lookback

In the next notebook, the three environments seen in the series (Cart Pole, Lunar Lander and Car Racing) will be tested with a Photonic PPO.